In [ ]:
import numpy as np
import pandas as pd
import cobra

def compute_metabolite_turnover(model, flux_series):
    """
    turnover_i = 0.5 * sum_j |S_ij * v_j|
    """
    rxn_ids = [rxn.id for rxn in model.reactions]
    met_ids = [met.id for met in model.metabolites]
    met_names = [met.name if met.name else met.id for met in model.metabolites]

    # Align flux values to reaction order
    flux_values = np.zeros(len(rxn_ids))
    for i, rxn_id in enumerate(rxn_ids):
        if rxn_id in flux_series.index:
            flux_values[i] = abs(flux_series[rxn_id])

    # Calculating turnover for each metabolite
    turnover_rates = []
    for met in model.metabolites:
        total_flux = 0.0
        for rxn in met.reactions:
            rxn_idx = rxn_ids.index(rxn.id)
            stoich_coeff = abs(rxn.metabolites[met])
            total_flux += stoich_coeff * flux_values[rxn_idx]
        turnover_rates.append(0.5 * total_flux)

    return pd.DataFrame({
        'met_id': met_ids,
        'met_name': met_names,
        'turnover': turnover_rates
    })

def turnover_from_fba(model):
    sol = model.optimize()
    if sol.status != 'optimal':
        raise RuntimeError(f'not optimal: {sol.status}')
    return compute_metabolite_turnover(model, sol.fluxes)

def add_tissue_tag(df):
    def tag(m):
        parts = m.split('_', 1)
        return parts[0] if len(parts) > 1 else 'NA'
    df = df.copy()
    df['tissue'] = df['met_id'].apply(tag)
    return df

model_PC = cobra.io.load_matlab_model('./models/new_models/MulModel_PC.mat')
model_CT = cobra.io.load_matlab_model('./models/new_models/MulModel_CT.mat')

print(f"PC model: {len(model_PC.reactions)} reactions, {len(model_PC.metabolites)} metabolites")
print(f"CT model: {len(model_CT.reactions)} reactions, {len(model_CT.metabolites)} metabolites")

# Calculating turnover
print("Calculating turnover for PC model...")
T_PC = turnover_from_fba(model_PC)

print("Calculating turnover for CT model...")
T_CT = turnover_from_fba(model_CT)

# Adding tissue tags
T_PC = add_tissue_tag(T_PC)
T_CT = add_tissue_tag(T_CT)

# Comparing only common metabolites
Comp = T_PC.merge(T_CT, on='met_id', suffixes=('_PC','_CT'), how='inner')
Comp['delta_PC_minus_CT'] = Comp['turnover_PC'] - Comp['turnover_CT']
Comp['fold_change'] = Comp['turnover_PC'] / (Comp['turnover_CT'] + 1e-10)  # Avoid division by zero

# Sorting by delta (PC - CT) highest positive first, then negative
Comp = Comp.sort_values('delta_PC_minus_CT', ascending=False)

results = pd.DataFrame({
    'met_id': Comp['met_id'],
    'met_name': Comp['met_name_PC'],
    'tissue': Comp['tissue_PC'],
    'ct_turnover': Comp['turnover_CT'],
    'pc_turnover': Comp['turnover_PC'],
    'delta_pc_ct': Comp['delta_PC_minus_CT'],
    'fold_change': Comp['fold_change']
})

results.to_csv('turnover_results_PC_vs_CT.csv', index=False)

# Getting top 50 highest turnover metabolites in PC condition
top_50_pc = results.nlargest(50, 'pc_turnover')
top_50_pc.to_csv('top_50_highest_turnover_PC.csv', index=False)

print(f"\nTotal metabolites compared: {len(results)}")
print(f"Metabolites with higher turnover in PC: {sum(results['delta_pc_ct'] > 0)}")
print(f"Metabolites with lower turnover in PC: {sum(results['delta_pc_ct'] < 0)}")
print(f"Metabolites with no change: {sum(results['delta_pc_ct'] == 0)}")

print("\nTop 15 metabolites with highest INCREASE in PC (positive delta):")
print(results.head(15).to_string(index=False))

print("\nTop 15 metabolites with highest DECREASE in PC (negative delta):")
print(results.tail(15).to_string(index=False))

print(f"   - turnover_results_PC_vs_CT.csv (all results)")
print(f"   - top_50_highest_turnover_PC.csv (top 50 highest PC turnover)")

ModuleNotFoundError: No module named 'cobra'

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
pip install cobra

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.0 MB ? eta -:--:--Downloading cobra-0.31.1-py2.py3-none-any.whl (1.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.5 MB/s eta 0:00:00


In [ ]:
import cobra

In [ ]:
def compute_metabolite_turnover(model, flux_series):
    """
    turnover_i = 0.5 * sum_j |S_ij * v_j|
    """
    rxn_ids = [rxn.id for rxn in model.reactions]
    met_ids = [met.id for met in model.metabolites]
    met_names = [met.name if met.name else met.id for met in model.metabolites]

    # Align flux values to reaction order
    flux_values = np.zeros(len(rxn_ids))
    for i, rxn_id in enumerate(rxn_ids):
        if rxn_id in flux_series.index:
            flux_values[i] = abs(flux_series[rxn_id])

    # Calculating turnover for each metabolite
    turnover_rates = []
    for met in model.metabolites:
        total_flux = 0.0
        for rxn in met.reactions:
            rxn_idx = rxn_ids.index(rxn.id)
            stoich_coeff = abs(rxn.metabolites[met])
            total_flux += stoich_coeff * flux_values[rxn_idx]
        turnover_rates.append(0.5 * total_flux)

    return pd.DataFrame({
        'met_id': met_ids,
        'met_name': met_names,
        'turnover': turnover_rates
    })

def turnover_from_fba(model):
    sol = model.optimize()
    if sol.status != 'optimal':
        raise RuntimeError(f'not optimal: {sol.status}')
    return compute_metabolite_turnover(model, sol.fluxes)



model_PC = cobra.io.load_matlab_model('/content/mkn28_6_updated.mat')
model_CT = cobra.io.load_matlab_model('/content/mkn28_0_updated.mat')

m1=input("Enter the name of the compared model:")
m2=input("Enter the name of the control model:")

print(f"{m1}: {len(model_PC.reactions)} reactions, {len(model_PC.metabolites)} metabolites")
print(f"{m2}: {len(model_CT.reactions)} reactions, {len(model_CT.metabolites)} metabolites")

# Calculating turnover
print(f"Calculating turnover for {m2} model...")
T_PC = turnover_from_fba(model_PC)

print(f"Calculating turnover for {m2} model...")
T_CT = turnover_from_fba(model_CT)

# Adding tissue tags
#T_PC = add_tissue_tag(T_PC)
#T_CT = add_tissue_tag(T_CT)

# Comparing only common metabolites
Comp = T_PC.merge(T_CT, on='met_id', suffixes=(f'_{m1}',f'_{m2}'), how='inner')
Comp[f'delta_{m1}_minus_{m2}'] = Comp[f'turnover_{m1}'] - Comp[f'turnover_{m2}']
Comp['fold_change'] = Comp[f'turnover_{m1}'] / (Comp[f'turnover_{m2}'] + 1e-10)  # Avoid division by zero

# Sorting by delta (PC - CT) highest positive first, then negative
Comp = Comp.sort_values(f'delta_{m1}_minus_{m2}', ascending=False)

results = pd.DataFrame({
    'met_id': Comp['met_id'],
    'met_name': Comp[f'met_name_{m1}'],
    f'{m2}_turnover': Comp[f'turnover_{m2}'],
    f'{m1}_turnover': Comp[f'turnover_{m1}'],
    f'delta_{m1}_{m2}': Comp[f'delta_{m1}_minus_{m2}'],
    'fold_change': Comp['fold_change']
})

results.to_csv(f'turnover_results_{m1}_vs_{m2}.csv', index=False)

# Getting top 50 highest turnover metabolites in PC condition
top_50_pc = results.nlargest(100, f'{m1}_turnover')
top_50_pc.to_csv(f'top_50_highest_turnover_{m1}.csv', index=False)

print(f"\nTotal metabolites compared: {len(results)}")
print(f"Metabolites with higher turnover in {m1}: {sum(results[f'delta_{m1}_{m2}'] > 0)}")
print(f"Metabolites with lower turnover in {m1}: {sum(results[f'delta_{m1}_{m2}'] < 0)}")
print(f"Metabolites with no change: {sum(results[f'delta_{m1}_{m2}'] == 0)}")

print(f"\nTop 15 metabolites with highest INCREASE in {m1} (positive delta):")
print(results.head(15).to_string(index=False))

print(f"\nTop 15 metabolites with highest DECREASE in {m1} (negative delta):")
print(results.tail(15).to_string(index=False))

print(f"   - turnover_results_{m1}_vs_{m2}.csv (all results)")
print(f"   - top_50_highest_turnover_{m1}.csv (top 50 highest PC turnover)")

Enter the name of the compared model:MKN28_6
Enter the name of the control model:MKN28_0
MKN28_6: 7555 reactions, 4793 metabolites
MKN28_0: 7666 reactions, 4804 metabolites
Calculating turnover for MKN28_0 model...
Calculating turnover for MKN28_0 model...

Total metabolites compared: 4762
Metabolites with higher turnover in MKN28_6: 502
Metabolites with lower turnover in MKN28_6: 409
Metabolites with no change: 3851

Top 15 metabolites with highest INCREASE in MKN28_6 (positive delta):
    met_id                      met_name  MKN28_0_turnover  MKN28_6_turnover  delta_MKN28_6_MKN28_0  fold_change
      h[c]                        Proton         40.212438         53.040130              12.827692 1.318998e+00
      h[m]                        Proton         34.812596         45.254283              10.441687 1.299940e+00
    nh4[c]                      Ammonium          7.030684         16.296642               9.265958 2.317931e+00
    h2o[c]                         Water         25.5606

In [ ]:
# Finding the common between the two
ges_dat=pd.read_csv("/content/turnover_results_GES1_24_vs_GES1_0.csv")
mkn_dat=pd.read_csv("/content/turnover_results_MKN28_6_vs_MKN28_0.csv")

In [ ]:
mkn_dat

In [ ]:
common_metabolites=mkn_dat[mkn_dat['met_id'].isin(ges_dat['met_id'])]
common_metabolites

,met_id,met_name,MKN28_0_turnover,MKN28_6_turnover,delta_MKN28_6_MKN28_0,fold_change
0,val_L[c],L-Valine,0.180000,9.975356,9.795356,5.541864e+01
1,3mob[m],3-Methyl-2-Oxobutanoate,0.000000,9.735356,9.735356,9.735356e+10
2,val_L[m],L-Valine,0.000000,9.735356,9.735356,9.735356e+10
3,3mob[c],3-Methyl-2-Oxobutanoate,0.010000,9.735356,9.725356,9.735356e+02
4,gthrd[c],Reduced Glutathione,0.051031,8.346828,8.295797,1.635637e+02
...,...,...,...,...,...,...
4757,mal_L[c],(S)-Malate,16.712834,8.548643,-8.164191,5.115017e-01
4758,3mop[c],3-Methyl-2-Oxopentanoate,8.803551,0.010000,-8.793551,1.135905e-03
4759,3mop[m],3-Methyl-2-Oxopentanoate,8.803551,0.000000,-8.803551,0.000000e+00
4760,ile_L[m],L-Isoleucine,8.803551,0.000000,-8.803551,0.000000e+00


In [ ]:
top=common_metabolites.head(20)['met_name']
top

,met_name
0,L-Valine
1,3-Methyl-2-Oxobutanoate
2,L-Valine
3,3-Methyl-2-Oxobutanoate
4,Reduced Glutathione
5,Reduced Glutathione
6,2-Oxoglutarate
7,2-Oxoglutarate
8,"5,10-Methylenetetrahydrofolate"
9,Folate


In [ ]:
bottom=common_metabolites.tail(20)['met_name']
bottom

,met_name
4742,Nicotinamide Adenine Dinucleotide Phosphate - ...
4743,Nicotinamide Adenine Dinucleotide Phosphate
4744,Adenosine Triphosphate
4745,Adenosine Diphosphate
4746,Nicotinamide Adenine Dinucleotide - Reduced
4747,Nicotinamide Adenine Dinucleotide
4748,Chloride
4749,Chloride
4750,Proton
4751,Leukotriene B4


In [ ]:
table = pd.DataFrame({
    'Top 20 Metabolites': common_metabolites.head(20)['met_name'].values,
    'Bottom 20 Metabolites': common_metabolites.tail(20)['met_name'].values
})

print(table.to_string(index=False))

            Top 20 Metabolites                                 Bottom 20 Metabolites
                      L-Valine Nicotinamide Adenine Dinucleotide Phosphate - Reduced
       3-Methyl-2-Oxobutanoate           Nicotinamide Adenine Dinucleotide Phosphate
                      L-Valine                                Adenosine Triphosphate
       3-Methyl-2-Oxobutanoate                                 Adenosine Diphosphate
           Reduced Glutathione           Nicotinamide Adenine Dinucleotide - Reduced
           Reduced Glutathione                     Nicotinamide Adenine Dinucleotide
                2-Oxoglutarate                                              Chloride
                2-Oxoglutarate                                              Chloride
5,10-Methylenetetrahydrofolate                                                Proton
                        Folate                                        Leukotriene B4
                        Folate                                 12

In [ ]:
table.to_csv('metabolites_table.csv', index=False)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
d60=pd.read_csv("/content/fsr_mapping_MKN28_6th__Vs__MKN28_0thdownregulated_raw_file (1).csv")
u60=pd.read_csv("/content/fsr_mapping_MKN28_6th__Vs__MKN28_0thupregulated_raw_file (1).csv")
d62=pd.read_csv("/content/fsr_mapping_MKN28_6th__Vs__MKN28_2nddownregulated_raw_file (1).csv")
u62=pd.read_csv("/content/fsr_mapping_MKN28_6th__Vs__MKN28_2ndupregulated_raw_file (1).csv")
u20=pd.read_csv("/content/fsr_mapping_MKN28_2nd__Vs__MKN28_0thupregulated_raw_file (1).csv")
d20=pd.read_csv("/content/fsr_mapping_MKN28_2nd__Vs__MKN28_0thdownregulated_raw_file (1).csv")


In [ ]:
unique_to_df1 = u20[~u20['Reaction Names'].isin(u62['Reaction Names'])]

# Get reactions unique to df2 (not present in df1)
unique_to_df2 = u62[~u62['Reaction Names'].isin(u20['Reaction Names'])]
unique_to_df1

,ReactionID,minFlux_control,maxFlux_control,FluxSpan_control,minFlux_infected,maxFlux_infected,FluxSpan_infected,FSR,Interpretation,Reaction Names,Subsystem,GPR,GPR_new,Entrez_IDs,Gene_Symbols
0,CHSTEROLt1,0.000000,0.000001,0.000001,-1000.000000,1000.000000,2000.000000,1.448852e+09,Upregulated,Cholesterol Intracellular Transport,"Transport, mitochondrial",10948.1,STARD3,['10948'],STARD3
1,ALLOP1tu,0.000000,0.000005,0.000005,0.000000,1000.000000,1000.000000,2.053099e+08,Upregulated,uptake of allopurinol by the enterocytes,Drug metabolism,9154.1 or 9153.1 or 64078.1,SLC28A1 or SLC28A2 or SLC28A3,"['64078', '9153', '9154']","SLC28A3 , SLC28A2 , SLC28A1"
2,r0812,0.000000,0.000013,0.000013,-1000.000000,1000.000000,2000.000000,1.517809e+08,Upregulated,Mitochondrial Carrier (Mc) Tcdb:2.A.29.20.1,Purine catabolism,10478.1,SLC25A17,['10478'],SLC25A17
4,C30CPT1,-0.000017,0.000000,0.000017,-999.990035,1000.000000,1999.990035,1.183626e+08,Upregulated,Production of Propionylcarnitine,Fatty acid oxidation,1374.1 or 1375.1 or 126129.1,CPT1A or CPT1B or CPT1C,"['126129', '1374', '1375']","CPT1C , CPT1A , CPT1B"
5,DALAt2r,0.009975,0.010000,0.000025,-1000.000000,1000.000000,2000.000000,7.890888e+07,Upregulated,D-Alanine Transport via Proton Symport,"Transport, extracellular",206358.1,SLC36A1,['206358'],SLC36A1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
543,FAOXC163C164Gm,0.000000,0.000004,0.000004,0.000000,0.000007,0.000007,2.023767e+00,Upregulated,"Fatty Acid Beta Oxidation (C16:3->C16:4), Mito...",Fatty acid oxidation,37.1,ACADVL,['37'],ACADVL
544,FAOXC163Gm,0.000000,0.000004,0.000004,0.000000,0.000007,0.000007,2.023767e+00,Upregulated,"Isomerization (C16:3), Mitochondrial",Fatty acid oxidation,1632.1,ECI1,['1632'],ECI1
545,FAOXC163GC142m,0.000000,0.000004,0.000004,0.000000,0.000007,0.000007,2.023767e+00,Upregulated,"Fatty Acid Beta Oxidation (C16:3->C14:2), Mito...",Fatty acid oxidation,3030.1 and 3032.1,HADHA and HADHB,"['3030', '3032']","HADHA , HADHB"
546,PPAm,0.000000,0.096021,0.096021,0.000000,0.193509,0.193509,2.015278e+00,Upregulated,Inorganic Diphosphatase,Oxidative phosphorylation,27068.4 or 27068.1 or 27068.3 or 27068.2,PPA2 or PPA2 or PPA2 or PPA2,['27068'],PPA2


In [ ]:
unique_to_df1[['Reaction Names','Subsystem','Gene_Symbols']].sort_values(by='Subsystem', ascending=True).to_csv("up2v0.csv")


In [ ]:
unique_to_df2[['Reaction Names','Subsystem','Gene_Symbols']].sort_values(by='Subsystem', ascending=True).to_csv("up6v2.csv")

In [ ]:
unique_to_df1d = d62[~d62['Reaction Names'].isin(d20['Reaction Names'])]

# Get reactions unique to df2 (not present in df1)
unique_to_df2d = d62[~d62['Reaction Names'].isin(d20['Reaction Names'])]

In [ ]:
unique_to_df1d[['Reaction Names','Subsystem','Gene_Symbols']].sort_values(by='Subsystem', ascending=True).to_csv("down2v0.csv")
unique_to_df2d[['Reaction Names','Subsystem','Gene_Symbols']].sort_values(by='Subsystem', ascending=True).to_csv("down6v2.csv")

In [ ]:
unique_to_df1d = d60[~d60['Reaction Names'].isin(d62['Reaction Names'])]

# Get reactions unique to df2 (not present in df1)
unique_to_df2d = d62[~d62['Reaction Names'].isin(d60['Reaction Names'])]